# ScoreIA — Machine Learning Pipeline Specification

Versão: 5.5
Modo: Continuidade Privada Controlada

---

# 1. Objetivo do Documento

Este documento define formalmente:

* A arquitetura do pipeline de ML
* Estratégia de divisão dos dados
* Processo de otimização
* Regras estatísticas obrigatórias
* Fluxo de treino, validação e teste
* Estrutura de retorno dos resultados
* Garantias contra data leakage

Este documento é normativo.
Qualquer alteração deve gerar uma nova versão.

---

# 2. Escopo do Módulo de ML

O módulo de ML do ScoreIA é responsável por:

* Preparação estruturada de dados
* Separação estatística correta
* Otimização de hiperparâmetros
* Comparação de modelos
* Treino final
* Avaliação em teste
* Interpretabilidade
* (Futuro) Exportação
* (Futuro) Relatório técnico automático

---

# 3. Arquitetura do Pipeline

## 3.1 Componentes Envolvidos

* `data_pipeline.py`
* `model_factory.py`
* `optimization.py`
* `metrics.py`
* `interpretability.py`
* `agent.py` (apenas orquestração)

---

# 4. Estratégia de Divisão dos Dados

## 4.1 Regra Fundamental

O conjunto de teste é isolado antes de qualquer otimização.

---

## 4.2 Fluxo Oficial

```id="m2af1z"
Dataset original
    ↓
Holdout (ex: 80% treino / 20% teste)
    ↓
Treino
    ↓
Cross-validation (k folds)
    ↓
Otimização (Optuna)
    ↓
Treino final (com melhores parâmetros)
    ↓
Avaliação no teste
```

---

## 4.3 Regras Obrigatórias

1. Teste nunca participa da:

   * Imputação (fit)
   * Encoding (fit)
   * Escalonamento (fit)
   * Cross-validation
   * Otimização
   * Early stopping

2. Transformações ajustadas no treino são aplicadas no teste apenas via `.transform()`.

3. Cross-validation utiliza exclusivamente o conjunto de treino.

---

# 5. Engenharia de Dados

Responsável: `data_pipeline.py`

## 5.1 Etapas

* Separação X e y
* Split holdout
* Identificação de variáveis numéricas e categóricas
* Imputação
* Encoding
* Escalonamento (quando aplicável)

---

## 5.2 Transformações

```id="8ekq0x"
Treino:
    fit imputador
    fit encoder
    fit scaler
    transform treino

Teste:
    transform com objetos já ajustados
```

---

# 6. Cross-Validation

## 6.1 Tipos

* Regressão → KFold
* Classificação → StratifiedKFold

## 6.2 Parâmetros

* n_splits = 5 (default)
* shuffle = True
* random_state = 42

---

## 6.3 Estrutura do Loop

Para cada trial do Optuna:

```id="e7c5qf"
para cada fold:
    cria modelo
    treina no fold treino
    avalia no fold validação
calcula média das métricas
retorna média
```

A métrica retornada ao Optuna é a média dos folds.

---

# 7. Otimização

Responsável: `optimization.py`

Framework: Optuna

---

## 7.1 Modelos Suportados

* LightGBM
* MLP (PyTorch)
* Modo AUTO (comparação entre ambos)

---

## 7.2 Direção da Otimização

* Regressão → minimizar (ex: RMSE)
* Classificação → maximizar (ex: F1)

---

## 7.3 Early Stopping

* MLP → Early stopping manual
* LightGBM → eval_set quando aplicável

Early stopping ocorre apenas dentro do fold.

---

# 8. Modo AUTO

Se `modelo="auto"`:

```id="s9vt33"
Otimiza LightGBM
Otimiza MLP
Compara best_score (CV)
Seleciona melhor
Treina final no treino completo
Avalia no teste
```

Critério de escolha:

Exclusivamente a métrica de validação cruzada.

---

# 9. Treino Final

Após selecionar os melhores hiperparâmetros:

1. Modelo é treinado usando 100% do conjunto de treino.
2. Nenhuma informação do teste é utilizada.
3. Avaliação ocorre uma única vez no teste.

---

# 10. Métricas

Responsável: `metrics.py`

## 10.1 Regressão

Exemplos:

* RMSE
* MAE
* R²

## 10.2 Classificação

Exemplos:

* F1-score
* Accuracy
* Precision
* Recall

Separação formal entre:

* Métrica de CV
* Métrica de teste

---

# 11. Estrutura de Retorno do Pipeline

O retorno final da execução de ML deve conter:

```id="ul82sx"
{
    "melhor_modelo": {
        "nome": "...",
        "best_params": {...},
        "best_score_cv": ...
    },
    "modelo_final": objeto_modelo_treinado,
    "metricas_teste": {...},
    "metadata": {...}
}
```

---

# 12. Garantias Estatísticas

O pipeline assegura:

* Ausência de data leakage
* Separação clara entre treino e teste
* Comparação justa entre modelos
* Reprodutibilidade via random_state
* Modularidade e extensibilidade

---

# 13. Pendências da Fase 5.5

Ainda não implementado:

1. SHAP + Feature Importance
2. Exportação do modelo
3. Relatório técnico automático (PDF)

Esses componentes deverão:

* Não interferir na lógica central
* Não modificar regras de split
* Ser implementados como módulos independentes

---

# 14. Restrições Arquiteturais

1. agent.py não deve conter lógica estatística.
2. Webapp não deve conter lógica de modelagem.
3. Split holdout sempre precede CV.
4. Teste é utilizado apenas para avaliação final.
5. Alterações no pipeline exigem atualização deste documento.

---

# 15. Estado Oficial

Versão ativa: 5.5
Pipeline estatisticamente consistente.
Pronto para expansão com interpretabilidade e exportação.

---

Se quiser, agora podemos criar:

* `CODEX_CONTEXT_RULES.ipynb` (para evitar que o Codex altere indevidamente o pipeline)
* Ou um plano técnico detalhado para implementar SHAP, exportação e relatório PDF de forma modular

Qual prefere construir agora?


## Atualizacao Fase 5.5 - Interpretabilidade Implementada

### O que foi adicionado
- Novo modulo: `tools/interpretability.py`
- Integracao no `core/agent.py::analisar_dados()` para gerar interpretabilidade no fim do treino
- Exportacao automatica em JSON no diretorio `reports/`

### Regras de classificacao aplicadas
- O threshold de classificacao usado na interpretabilidade e o mesmo definido/otimizado na modelagem
- Nao usamos cutoff fixo de 0.5, exceto fallback tecnico quando nao ha threshold valido

### Metodos por tecnica
- LightGBM: `feature_importances_` + SHAP (quando pacote `shap` estiver disponivel)
- MLP: Permutation Importance + ALE (efeito marginal local acumulado)

### Parametros de configuracao no pipeline
- `threshold_classificacao`
- `interpretability_top_n`
- `interpretability_n_repeats`
- `interpretability_ale_top_k`
- `interpretability_shap_sample_size`
- `reports_dir`
